In [ ]:
# Lab type: debug
# Course: AI402 — Retrieval & RAG Systems
# Lesson: Chunking Strategies: The Decision AI Tools Get Wrong
# Task: An AI assistant wrote the ingestion pipeline below. It contains 3 bugs
# that silently destroy retrieval quality. Find and fix each one, and write a
# one-sentence explanation in the markdown cell after each fix.

# Lab: Debugging an AI-Generated Chunking Pipeline

A team asked an AI assistant to "chunk and index our knowledge base". The pipeline below runs without errors and most queries work — but tail queries fail, and they fail silently.

**Your task:** find the 3 bugs, fix them, and verify the fix by comparing recall on the labelled query set before and after.

**Outputs are cleared.** Run every cell top to bottom.

## Setup

In [ ]:
!pip install sentence-transformers rank-bm25 faiss-cpu numpy pandas --quiet

In [ ]:
import numpy as np

# The Nimbus Analytics product knowledge base: (doc_id, heading_path, text)
CORPUS = [
    ("plans-overview", "Pricing > Plans",
     "Nimbus Analytics offers three subscription plans: Starter, Teams, and "
     "Enterprise. Starter includes 5 seats and community support. Teams includes "
     "50 seats, shared dashboards, and priority email support. Enterprise includes "
     "unlimited seats, priority support, and advanced security features."),
    ("sso-policy", "Pricing > Enterprise plan",
     "Single sign-on (SSO) with SAML 2.0 is available on the Enterprise plan only. "
     "The Teams plan does not include SSO. Enterprise customers can configure SSO "
     "from the admin console under Security settings."),
    ("seat-pricing", "Pricing > Seats",
     "Per-seat pricing: Starter is $12 per seat per month, Teams is $29 per seat "
     "per month, and Enterprise pricing is custom. Annual billing gives a 20 "
     "percent discount on all plans."),
    ("refund-policy", "Billing > Refunds",
     "Customers can request a full refund within 30 days of purchase. To get your "
     "money back after 30 days, contact billing support; partial refunds are "
     "prorated for annual subscriptions."),
    ("error-e4022", "Troubleshooting > Error codes",
     "Error E4022 means the API rate limit was exceeded. The Starter plan allows "
     "100 requests per minute, Teams 1,000, and Enterprise 10,000. Wait 60 seconds "
     "and retry, or upgrade the plan."),
    ("error-e5001", "Troubleshooting > Error codes",
     "Error E5001 indicates an expired API token. Rotate the token from the admin "
     "console under API settings. Tokens expire after 90 days by default."),
    ("api-export", "API > Export",
     "The export endpoint POST /v2/export creates a CSV export of dashboard data. "
     "Exports are limited to 100,000 rows on Teams and 1 million rows on "
     "Enterprise."),
    ("data-retention", "Security > Data retention",
     "Event data is retained for 13 months on all plans. Enterprise customers can "
     "configure custom retention windows up to 5 years from the admin console."),
    ("priority-support", "Support > Tiers",
     "Priority support with a 4-hour response SLA is included in Teams and "
     "Enterprise plans. Starter includes community support only."),
    ("dashboard-sharing", "Product > Dashboards",
     "Shared dashboards let teammates view and edit the same dashboard. Sharing "
     "outside your workspace requires a public link, available on Teams and "
     "Enterprise."),
    ("audit-logs", "Security > Audit logs",
     "Audit logs record sign-ins, permission changes, and data exports. Audit "
     "logs are an Enterprise-only feature and are retained for 2 years."),
    ("cancel-downgrade", "Billing > Cancellation",
     "You can cancel or downgrade at any time from the billing page. Downgrades "
     "take effect at the end of the current billing period."),
]
DOC_IDS = [d[0] for d in CORPUS]
DOC_TEXTS = [f"{d[1]}: {d[2]}" for d in CORPUS]

# Labelled evaluation queries: (query, set of relevant doc_ids)
EVAL_SET = [
    ("does the teams plan include sso", {"sso-policy"}),
    ("how do I get my money back", {"refund-policy"}),
    ("what does error E4022 mean", {"error-e4022"}),
    ("how long is event data kept", {"data-retention"}),
    ("cost per seat on the teams plan", {"seat-pricing"}),
    ("response time for priority support", {"priority-support"}),
    ("row limit for csv export", {"api-export"}),
    ("rotate an expired api token", {"error-e5001"}),
]
print(f"{len(CORPUS)} documents, {len(EVAL_SET)} labelled queries")

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed(texts):
    return embedder.encode(list(texts), normalize_embeddings=True)

DOC_EMB = embed(DOC_TEXTS)

def dense_search(query, k=5, doc_emb=None, doc_ids=None):
    doc_emb = DOC_EMB if doc_emb is None else doc_emb
    doc_ids = DOC_IDS if doc_ids is None else doc_ids
    scores = doc_emb @ embed([query])[0]
    order = np.argsort(scores)[::-1][:k]
    return [(doc_ids[i], float(scores[i])) for i in order]

print(dense_search("does the teams plan include sso", k=3))

## The AI-generated ingestion pipeline

Read it the way you would review a pull request. It chunks every document, embeds the chunks, and provides a search function. Three of its decisions are production defects.

In [ ]:
# --- AI-GENERATED PIPELINE (contains 3 bugs) ---
# Review this code — is it correct?

def chunk_document(text, size=120):
    # Bug candidate area 1: how are boundaries chosen?
    return [text[i:i + size] for i in range(0, len(text), size)]

chunks = []
for doc_id, heading, text in CORPUS:
    # Bug candidate area 2: what does each chunk carry with it?
    for piece in chunk_document(text):
        chunks.append(piece)

chunk_emb = embed(chunks)

def chunk_search(query, k=5):
    scores = chunk_emb @ embed([query])[0]
    order = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in order]

print(f"{len(chunks)} chunks indexed")
for text, score in chunk_search('does the teams plan include sso', k=3):
    print(f"  {score:.3f}  {text[:70]!r}")

## Measure before you fix

A RAG pipeline is only as good as the *facts* its chunks deliver intact. For six queries we know the exact answer-bearing span in the source document; the measurement below checks whether any retrieved chunk contains that span **unbroken**. (The pipeline returns bare text with no document identity, so span matching is also all we *can* measure — that awkwardness is itself a clue to one of the bugs.)

In [ ]:
# (query, answer-bearing span that a retrieved chunk must contain intact)
ANSWER_SPANS = [
    ("does the teams plan include sso",
     "The Teams plan does not include SSO"),
    ("how do I get my money back",
     "request a full refund within 30 days"),
    ("can I keep event data longer than 13 months",
     "custom retention windows up to 5 years"),
    ("row limit for csv export on enterprise",
     "100,000 rows on Teams and 1 million rows on Enterprise"),
    ("api rate limits per plan",
     "100 requests per minute, Teams 1,000, and Enterprise 10,000"),
    ("response time for priority support",
     "4-hour response SLA"),
]

def intact_span_rate(search_fn, k=3):
    hits = 0
    for query, span in ANSWER_SPANS:
        retrieved_texts = [chunk for chunk, _ in search_fn(query, k=k)]
        if any(span in text for text in retrieved_texts):
            hits += 1
        else:
            print(f"  MISSED intact: {span!r:.60}")
    return hits / len(ANSWER_SPANS)

print(f"Buggy pipeline — answer spans delivered intact @3: "
      f"{intact_span_rate(chunk_search):.2f}")

## Bug 1

Look at `chunk_document` and at a few actual chunks (`chunks[:5]`). What do the boundaries fall in the middle of?

**Explain the bug:**

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**The bug:** `chunk_document` splits on raw *character* positions with no overlap. Boundaries fall mid-word and mid-sentence, so facts spanning a boundary exist intact in no chunk at all.

**Why it causes wrong behaviour:** a chunk like `'gle sign-on (SSO) with SAML 2.0 is availab'` embeds to a vector that no longer resembles queries about SSO availability — the content silently becomes unreachable for exactly the queries it answers.

**Correct approach:** split on sentence boundaries with overlap, or — better for this corpus — keep each document's text intact per section and carry the heading path (see Bug 3 fix).

</details>

## Bug 2

What does each entry in `chunks` carry besides its text? Think about what you would need to (a) update the index when a document changes, (b) cite a source, (c) filter by tenant.

**Explain the bug:**

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**The bug:** chunks are stored as bare strings — the `doc_id` and heading path are thrown away at ingestion.

**Why it causes wrong behaviour:** without per-chunk metadata you cannot delete a changed document's chunks (updates), attribute an answer to a source (citations), or enforce access control (permissions). The eval harness above had to fall back to substring matching precisely because chunk→document identity was lost.

**Correct approach:** store `(doc_id, heading, chunk_text)` triples (or a dict) and embed the text while keeping the identity attached.

</details>

## Bug 3

The corpus documents carry a heading path like `'Pricing > Enterprise plan'`. Where does that information go in the pipeline, and what happens to a chunk whose meaning depends on it?

**Explain the bug:**

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**The bug:** the heading path is dropped — chunks are embedded without the structural context that says what they are *about*.

**Why it causes wrong behaviour:** a sentence like "...is available on the Enterprise plan only" chunked away from its `Pricing` heading loses the vocabulary that queries actually use. Structure-aware chunking exists to carry exactly this context.

**Correct approach:** prepend the heading path to each chunk's text before embedding, as the fixed pipeline below does.

</details>

## The fixed pipeline

All three fixes applied. Run it and compare recall.

In [ ]:
# Fix: sentence-boundary chunks + carried metadata + heading path prepended
import re

def chunk_document_fixed(text, max_sentences=2, overlap=1):
    sentences = re.split(r"(?<=[.!?]) +", text)
    step = max(1, max_sentences - overlap)
    return [" ".join(sentences[i:i + max_sentences])
            for i in range(0, len(sentences), step)]

fixed_chunks = []          # (doc_id, heading, chunk_text)
for doc_id, heading, text in CORPUS:
    for piece in chunk_document_fixed(text):
        fixed_chunks.append((doc_id, heading, piece))

fixed_emb = embed([f"{h}: {t}" for _, h, t in fixed_chunks])

def chunk_search_fixed(query, k=5):
    scores = fixed_emb @ embed([query])[0]
    order = np.argsort(scores)[::-1][:k]
    return [(fixed_chunks[i][2], float(scores[i])) for i in order]

print(f"Fixed pipeline — answer spans delivered intact @3: "
      f"{intact_span_rate(chunk_search_fixed):.2f}")

# And because metadata now exists, honest doc-level recall is measurable:
def doc_search_fixed(query, k=3):
    scores = fixed_emb @ embed([query])[0]
    order = np.argsort(scores)[::-1][:k]
    return [fixed_chunks[i][0] for i in order]

hits = sum(1 for q, rel in EVAL_SET if set(doc_search_fixed(q)) & rel)
print(f"Fixed pipeline recall@3 by doc_id (impossible before Bug 2's fix): "
      f"{hits / len(EVAL_SET):.2f}")

## Summary

Fill in the blanks, then check your answers below.

1. Character-window chunking with no overlap makes boundary-spanning facts _______ without any error surfacing.
2. Chunk metadata (doc_id, heading path) must be captured at _______ time — afterwards it cannot be recovered from the index.
3. Prepending the _______ to each chunk's text keeps small chunks self-describing.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **unreachable/unretrievable** — the chunks embed away from the queries they answer.
2. **ingestion** — updates, citations, and permissions all depend on it.
3. **heading path** — structure-aware chunking carries the context a boundary would destroy.

</details>